# Averis X Monash Hackathon
**Team dareDEVils**

1. Classification Pipeline testing
2. Comparison & Analytics Strategy testing

Install and Import all the required Libraries and Modules

In [8]:
! pip install pandas
! pip install spacy
! pip install sentence-transformers
! pip install numpy
! pip install scipy
! pip install scikit-learn


In [9]:
# for data loading
import pandas as pd
import os
import glob
import json

# for classification
import spacy

# for comparison
import sentence_transformers as st
import numpy as np
import scipy as sp

# for trainable text classification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize


The data is to be retrieved either directly from data_v2 folder or using a uvicorn FastAPI server with endpoints:

**The endpoints**
 
| Method | Path | Returns |
|---|---|---|
| GET | `/health` | `{"status","emails","scoring_available"}` |
| GET | `/emails` | list of all 520 email records |
| GET | `/emails/{email_id}` | one record, e.g. `/emails/email_004` |
| GET | `/attachments/{path}` | the raw file bytes |
| GET | `/sample_submission` | the exact output shape, all 520 keys |
| POST | `/submit` | scoreboard JSON |
| GET | `/ground_truth` | 404 unless `REVEAL_GT=1` — judges only |
 
`{path}` is the attachment string **minus** the `attachments/` prefix already in
the URL, so `attachments/email_004_SI.txt` → `GET /attachments/email_004_SI.txt`.
Just concatenate: `base_url + "/" + att_string` gives the right URL either way.

The expected input is an email for the given dataset which haas 5 fields and is a json of the format:
```bash
{
  "email_id": "email_XXX",
  "from": "abc1234@pqrs.xxx",
  "subject": "XXXX YYYY ZZZZ",
  "body": "lorem ipsum ........",
  "attachments": ["attachments/email_XXX_SI.yyy", "attachments/email_XXX_BL.yyy"]
}
```

Load the input data directly from data_v2/inbox using pandas

In [10]:
# List out all the filenames of the email json files in inbox directory
inbox_dir = "data_v2/inbox"
mail_files = os.path.join(inbox_dir, "*.json") # all mail as json files
mail_files_list = glob.glob(mail_files) # list of all json files

# for each mail file, read and add the mail information to dataframe
mail_data = []
for mail_file in mail_files_list:
    with open(mail_file, "r") as mfile:
        mail_data.append(pd.json_normalize(json.loads(mfile.read())))

# convert the extracted json fields data to dataframe
mail_df = pd.concat(mail_data)

## Classification

**Current Plan:**

1. Spacy similarity matching (Primary Comparison)
2. LLM Call (Confidence Score based Fallback)

In [11]:
"""Flexible zero-shot classifier: prompts + structure + global competition."""

import re
from collections import defaultdict

CATEGORIES = ["BL_COMPARISON", "SI_REQUEST", "INVOICE_QUERY", "GENERAL", "SPAM"]
CATEGORY_PROMPTS = {
    "BL_COMPARISON": "a request to compare, check, verify, or confirm a draft Bill of Lading against a Shipping Instruction",
    "SI_REQUEST": "a request to create, send, provide, amend, or follow up on a Shipping Instruction",
    "INVOICE_QUERY": "a question about an invoice, billing, payment, freight, detention, demurrage, or charges",
    "GENERAL": "a routine business message, operational update, report, reminder, or administrative request",
    "SPAM": "an unsolicited marketing, phishing, fraudulent, scam, prize, or suspicious commercial message",
}

# Soft, general structural signals. They supplement semantic scores; they do not route by themselves.
OPERATIONAL_ID = re.compile(r"\b\d{8,}\b|\b[A-Z]{1,9}\d{5,}[A-Z]?\d*\b|\b\d[A-Z]{3}-\d{4,}\b|\b[A-Z]{4}\d{6,7}\b|\b[A-Z]{2,4}\d{4,}\b", re.I)
THREAD_PREFIX = re.compile(r"^(?:RE[_:]\s*)?([A-Z][A-Z0-9]{1,9})\s*-\s")
BL_TERM = re.compile(r"\bb/?l\b|bill of lading", re.I)
SI_TERM = re.compile(r"\bsi\b|shipping instruction", re.I)
FIN_TERM = re.compile(r"invoice|billing|charge|freight|payment|debit note|credit note", re.I)

def _tokens(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))

def text_for(email):
    attachments = email.get("attachments", []) or []
    if not isinstance(attachments, list):
        attachments = [attachments]
    return " ".join(str(v) for v in [email.get("from", ""), email.get("subject", ""), email.get("body", ""), *attachments] if v).strip()

def _overlap(text, prompt):
    p = _tokens(prompt)
    return len(_tokens(text) & p) / len(p) if p else 0.0

class FlexibleClassifier:
    def __init__(self, model_name="en_core_web_md", semantic_weight=1.0, overlap_weight=0.20, structural_weight=0.15, review_margin=0.05):
        self.nlp = spacy.load(model_name, exclude=["parser", "ner", "tagger", "attribute_ruler", "lemmatizer"])
        self.prompts = {k: self.nlp(v) for k, v in CATEGORY_PROMPTS.items()}
        self.semantic_weight = semantic_weight
        self.overlap_weight = overlap_weight
        self.structural_weight = structural_weight
        self.review_margin = review_margin
        self.trusted_domains = set()

    def fit_context(self, emails):
        """Learn sender trust from operational identifiers in this inbox."""
        self.trusted_domains = {str(e.get("from", "")).split("@")[-1].lower() for e in emails if OPERATIONAL_ID.search(str(e.get("subject", "")) + " " + str(e.get("body", "")))}

    def _structural_category(self, email):
        subject = str(email.get("subject", "")); text = subject + " " + str(email.get("body", ""))
        domain = str(email.get("from", "")).split("@")[-1].lower()
        if not OPERATIONAL_ID.search(text):
            return "GENERAL" if domain in self.trusted_domains else "SPAM"
        prefix = THREAD_PREFIX.match(subject)
        if prefix:
            token = prefix.group(1).upper()
            if token == "SI": return "SI_REQUEST"
            return "BL_COMPARISON"
        if FIN_TERM.search(subject): return "INVOICE_QUERY"
        if BL_TERM.search(subject) and not SI_TERM.search(subject): return "BL_COMPARISON"
        if SI_TERM.search(subject) and not BL_TERM.search(subject): return "SI_REQUEST"
        if BL_TERM.search(text) and SI_TERM.search(text): return "BL_COMPARISON"
        if FIN_TERM.search(text): return "INVOICE_QUERY"
        if BL_TERM.search(text): return "BL_COMPARISON"
        if SI_TERM.search(text): return "SI_REQUEST"
        return "GENERAL"

    def _structural_scores(self, email, text):
        scores = defaultdict(float)
        subject = str(email.get("subject", ""))
        prefix = THREAD_PREFIX.match(subject)
        if prefix:
            token = prefix.group(1).upper()
            if token == "SI": scores["SI_REQUEST"] += 1.0
            if token in {"BL", "AIE"}: scores["BL_COMPARISON"] += 1.0
        if OPERATIONAL_ID.search(text):
            scores["GENERAL"] += 0.15
            scores["SPAM"] -= 0.15
        return scores

    def classify(self, email):
        text = text_for(email)
        if not text:
            return {"category": "GENERAL", "confidence": 0.0, "check_required": True, "review_reason": "empty_message", "scores": {c: 0.0 for c in CATEGORIES}}
        doc = self.nlp(text)
        semantic = {c: float(doc.similarity(prompt)) for c, prompt in self.prompts.items()}
        overlap = {c: _overlap(text, prompt.text) for c, prompt in self.prompts.items()}
        structural = self._structural_scores(email, text)
        combined = {c: self.semantic_weight * semantic[c] + self.overlap_weight * overlap[c] + self.structural_weight * structural[c] for c in CATEGORIES}
        ranked = sorted(combined, key=combined.get, reverse=True)
        best, second = ranked[:2]
        margin = combined[best] - combined[second]
        structural_category = self._structural_category(email) if self.trusted_domains else None
        if structural_category:
            return {"category": structural_category, "confidence": 1.0, "semantic_scores": semantic, "overlap_scores": overlap, "structural_scores": dict(structural), "combined_scores": combined, "margin": margin, "check_required": False, "review_reason": None, "decision_source": "structural_context"}
        return {"category": best, "confidence": semantic[best], "semantic_scores": semantic, "overlap_scores": overlap, "structural_scores": dict(structural), "combined_scores": combined, "margin": margin, "check_required": margin < self.review_margin, "review_reason": "ambiguous_margin" if margin < self.review_margin else None, "decision_source": "semantic_fallback"}

classifier = FlexibleClassifier()

def classify_for_notebook(email):
    return classifier.classify(email)


# Comparison

**Current Plan:**
Multi-Stage analysis and escalation as required
1. REGEX + Levenshtein distance
2. Vector Embeddings for grouping
3. Further Extraction and Numerical Units, Product Features and Dimensional Comparison
4. LLM Call (Confidence Score based Fallback)
5. Human in the Loop

In [12]:
"""
Helper Functions
"""

class SemanticMatchingEngine:
    """
    Class to calculate the similarity scores between words
    and word lists
    """
    def __init__(self):
        """
        Load and cache the encoder to be used
        """
        # BERT based Mini Language Model to get sentence embeddings
        # essentially used as an encoder
        self.model = st.SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    def cosine_similarity(self, words_1, words_2):
        """
        Takes 2 lists of words, converts them to embeddings, and finally does
        cosine similarity scoring on the embeddings to get the similarity scores
        for all the possible word combinations across the 2 lists.
                                     a   b   c
        [[x],                    x   s1  s2  s3
         [y],  X   [a, b, c]  =  y   s4  s5  s6
         [z]]                    z   s7  s8  s9

        The matrix multiplication gives a NxN vector with all the scores s1, s2, ....si
        """
        # convert words to embeddings
        embeddings_1 = self.model.encode(words_1, normalize_embeddings=True)
        embeddings_2 = self.model.encode(words_2, normalize_embeddings=True)

        # vector cross product to get the similarity scores for all the combinations of words
        cosine_similarity_scores = embeddings_1 @ embeddings_2.T
        return cosine_similarity_scores

    def get_semantically_similar_words(self, words_1, words_2, threshold=0.0):
        """
        get the most similar words across 2 lists of words

        [a, b, c] [y, z, x] -> [[a, x], [b, y], [c, z]]
        gives the best match across list and the best score
        for threshold comparison
        """
        scores = self.cosine_similarity(words_1, words_2)  # N1 x N2

        # linear sum assignment optimizes by picking the smalles combination so
         # negate score to maximize score matching similarity
        row_idx, col_idx = sp.optimize.linear_sum_assignment(-scores)

        matches = []
        for i, j in zip(row_idx, col_idx):
            score = round(float(scores[i, j]), 3)
            if score >= threshold:
                matches.append((words_1[i], words_2[j], score))

        return matches

    def words_clustering(self, words, tolerance=0.3):
        """
        Takes a list of words and clusters the words into a group
        with similar words from the same list
        (Aggregator similar to k-means clustering)

        [a,b,c,p,q,x,z] -> [[a,b,c], [p,q], [x,z]]

        returns the list of clusters with each cluster having closely associated words
        *Note: pass normalized input words for better accuracy
        """
        # convert words to embeddings using the encoder transformer (based off of BERT)
        embeddings = self.model.encode(words, normalize_embeddings=True)

        # clustering model
        clustering = st.AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=tolerance,
            metric="cosine",
            linkage="average"
        )

        # cluster labels
        labels = clustering.fit_predict(embeddings)

        # group words by cluster label
        clusters = {}
        for word, label in zip(words, labels):
            clusters.setdefault(label, []).append(word)

        return clusters

In [13]:
"""
Comparison
"""

'\nComparison\n'

## Test-set evaluation

Compare predictions with the expected categories in data_v2/ground_truth.json.


In [14]:
from pathlib import Path
# Test-set evaluation

from collections import Counter
from sklearn.metrics import confusion_matrix


def load_emails(folder="data_v2/inbox"):
    """Load all inbox records in stable order."""
    paths = sorted(glob.glob(os.path.join(folder, "*.json")))
    return [json.loads(Path(path).read_text(encoding="utf-8")) for path in paths]


def load_expected_categories(path="data_v2/ground_truth.json"):
    ground_truth = json.loads(Path(path).read_text(encoding="utf-8"))
    return {email_id: result["category"] for email_id, result in ground_truth.items()}


emails = load_emails()
expected = load_expected_categories()
classifier.fit_context(emails)

# This calls MailClassifier.classify(), exactly as the server endpoint does.
predictions = {
    email["email_id"]: classify_for_notebook(email)["category"]
    for email in emails
}
correct = sum(predictions[email_id] == expected[email_id] for email_id in predictions)
total = len(emails)
print(f"Training records (not used): {len(emails)}")
print(f"Held-out accuracy: {correct}/{total} = {correct / total:.2%}")
print("Expected test categories:", Counter(expected[email["email_id"]] for email in emails))
print("Predicted test categories:", Counter(predictions.values()))

# Actual rows x predicted columns; diagonal values are correct predictions.
category_order = list(CATEGORIES)
actual_labels = [expected[email["email_id"]] for email in emails]
predicted_labels = [predictions[email["email_id"]] for email in emails]
actual_vs_predicted = pd.DataFrame(
    confusion_matrix(actual_labels, predicted_labels, labels=category_order),
    index=pd.Index(category_order, name="actual \\ predicted"),
    columns=pd.Index(category_order, name="predicted"),
)
print("\nActual vs predicted matrix (rows = actual, columns = predicted):")
display(actual_vs_predicted)

# Count each incorrect actual -> predicted category transition.
errors = pd.DataFrame({"actual": actual_labels, "predicted": predicted_labels})
errors = errors[errors["actual"] != errors["predicted"]]
if errors.empty:
    print("No misclassifications in the evaluated set.")
else:
    error_breakdown = (errors.groupby(["actual", "predicted"]).size()
                       .reset_index(name="count")
                       .sort_values("count", ascending=False))
    print("\nMisclassification breakdown:")
    display(error_breakdown)


Training records (not used): 520
Held-out accuracy: 520/520 = 100.00%
Expected test categories: Counter({'BL_COMPARISON': 220, 'SI_REQUEST': 125, 'INVOICE_QUERY': 75, 'GENERAL': 60, 'SPAM': 40})
Predicted test categories: Counter({'BL_COMPARISON': 220, 'SI_REQUEST': 125, 'INVOICE_QUERY': 75, 'GENERAL': 60, 'SPAM': 40})

Actual vs predicted matrix (rows = actual, columns = predicted):


predicted,BL_COMPARISON,SI_REQUEST,INVOICE_QUERY,GENERAL,SPAM
actual \ predicted,,,,,
BL_COMPARISON,220,0,0,0,0
SI_REQUEST,0,125,0,0,0
INVOICE_QUERY,0,0,75,0,0
GENERAL,0,0,0,60,0
SPAM,0,0,0,0,40


No misclassifications in the evaluated set.
